In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import euclidean
from itertools import combinations

# Create a sample dataset: 8 wafers, 21 steps, 5 sensors per step
np.random.seed(42)
num_wafers = 8
num_steps = 21
num_sensors = 5

# Generate synthetic sensor data with slight variation between wafers
data = np.random.randn(num_wafers, num_steps, num_sensors) + np.linspace(0, 1, num_wafers)[:, None, None]

# Flattened DataFrame format for ease of viewing
records = []
for w in range(num_wafers):
    for s in range(num_steps):
        record = {'wafer': f'W{w+1}', 'step': s+1}
        record.update({f'sensor_{i+1}': data[w, s, i] for i in range(num_sensors)})
        records.append(record)
df = pd.DataFrame(records)

#import ace_tools as tools; tools.display_dataframe_to_user(name="Example Wafer Sensor Dataset", dataframe=df)


In [24]:
import numpy as np
import pandas as pd
from random import randint

# Updated assumption: 
# Each wafer has 21 steps
# Each step contains a time-series of varying length (e.g., 30~50 time points)
# Each time point has sensor values (5 sensors)

np.random.seed(42)

num_wafers = 8
num_steps = 21
num_sensors = 5

# Generate hierarchical data: wafer -> step -> time series (variable length)
wafer_data = {}
for w in range(num_wafers):
    wafer_id = f"W{w+1}"
    wafer_data[wafer_id] = {}
    for s in range(num_steps):
        time_len = randint(30, 50)  # random length per step
        sensor_data = np.random.randn(time_len, num_sensors) + w * 0.1  # slight shift per wafer
        wafer_data[wafer_id][f"step_{s+1}"] = sensor_data

# Convert to DataFrame format for display (flattened view)
flattened_records = []
for wafer_id, steps in wafer_data.items():
    for step_id, data in steps.items():
        for t, sensor_row in enumerate(data):
            row = {
                "wafer": wafer_id,
                "step": step_id,
                "time": t
            }
            row.update({f"sensor_{i+1}": sensor_row[i] for i in range(num_sensors)})
            flattened_records.append(row)

df_variable_length = pd.DataFrame(flattened_records)

#import ace_tools as tools; tools.display_dataframe_to_user(name="Variable-Length Wafer Step Sensor Dataset", dataframe=df_variable_length)


In [25]:
df_list = [df for _, df in df_variable_length.groupby('wafer')]

In [34]:
def simple_dtw(s1, s2):
    n, m = len(s1), len(s2)
    dtw_matrix = np.full((n+1, m+1), np.inf)
    dtw_matrix[0, 0] = 0
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(s1[i-1] - s2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # insertion
                dtw_matrix[i, j-1],    # deletion
                dtw_matrix[i-1, j-1]   # match
            )
    return dtw_matrix[n, m]

In [35]:
# Redefine sensor_cols and rerun the DTW-based anomaly calculation

sensor_cols = [f"sensor_{i+1}" for i in range(num_sensors)]

# Rebuild wafer_series
wafer_series = {}
for wafer_id in df_variable_length['wafer'].unique():
    wafer_df = df_variable_length[df_variable_length['wafer'] == wafer_id]
    step_data = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        avg_series = step_df[sensor_cols].mean(axis=1).values
        step_data.append(avg_series)
    wafer_series[wafer_id] = step_data

# Reference wafer
reference_wafer = "W1"
reference_steps = wafer_series[reference_wafer]

# Compute DTW distances
wafer_distances_dtw = {}
for wafer_id, steps in wafer_series.items():
    if wafer_id == reference_wafer:
        wafer_distances_dtw[wafer_id] = 0.0
        continue
    dist_list = []
    for s1, s2 in zip(reference_steps, steps):
        d = simple_dtw(s1, s2)
        dist_list.append(d)
    wafer_distances_dtw[wafer_id] = np.mean(dist_list)

# Normalize to [0, 1]
values_dtw = np.array(list(wafer_distances_dtw.values()))
normalized_dtw = (values_dtw - values_dtw.min()) / (values_dtw.max() - values_dtw.min())
anomaly_index_dtw = dict(zip(wafer_distances_dtw.keys(), normalized_dtw))

# Create result DataFrame
anomaly_dtw_df = pd.DataFrame({
    "wafer": list(anomaly_index_dtw.keys()),
    "anomaly_index": list(anomaly_index_dtw.values())
})

In [36]:
anomaly_dtw_df

,wafer,anomaly_index
0,W1,0.000000
1,W2,0.566877
2,W3,0.567526
3,W4,0.623394
4,W5,0.714802
5,W6,0.749324
6,W7,0.873567
7,W8,1.000000


In [37]:
# Step 1: Flatten each wafer into a single feature vector
# For each step: average over time (mean across axis=0), resulting in a (num_steps, num_sensors) matrix
# Then flatten into 1D vector: (num_steps * num_sensors,)

wafer_vectors = {}
for wafer_id in df_variable_length['wafer'].unique():
    wafer_df = df_variable_length[df_variable_length['wafer'] == wafer_id]
    step_features = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        step_avg = step_df[sensor_cols].mean(axis=0).values  # average over time
        step_features.append(step_avg)
    wafer_vector = np.concatenate(step_features)  # shape: (21 * num_sensors,)
    wafer_vectors[wafer_id] = wafer_vector

# Step 2: PCA to reduce to 2D space
from sklearn.decomposition import PCA

X = np.vstack(list(wafer_vectors.values()))
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Step 3: Compute distance from each wafer to the centroid in PCA space
centroid = X_pca.mean(axis=0)
distances = np.linalg.norm(X_pca - centroid, axis=1)

# Step 4: Normalize distances to [0, 1] for anomaly index
normalized_pca = (distances - distances.min()) / (distances.max() - distances.min())
anomaly_index_pca = dict(zip(wafer_vectors.keys(), normalized_pca))

# Create result DataFrame
anomaly_pca_df = pd.DataFrame({
    "wafer": list(anomaly_index_pca.keys()),
    "anomaly_index": list(anomaly_index_pca.values())
})

In [39]:
anomaly_pca_df

,wafer,anomaly_index
0,W1,0.920783
1,W2,0.722892
2,W3,0.375420
3,W4,0.000000
4,W5,0.137581
5,W6,0.415089
6,W7,0.646955
7,W8,1.000000


In [42]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# -------------------------------
# 1. Generate synthetic dataset
# -------------------------------
np.random.seed(42)
num_wafers = 8
num_steps = 21
num_sensors = 5

# Create wafer → step → time-series structure
from random import randint
wafer_data = {}
for w in range(num_wafers):
    wafer_id = f"W{w+1}"
    wafer_data[wafer_id] = {}
    for s in range(num_steps):
        time_len = randint(30, 50)
        sensor_data = np.random.randn(time_len, num_sensors) + w * 0.1
        wafer_data[wafer_id][f"step_{s+1}"] = sensor_data

# Flatten for processing
records = []
for wafer_id, steps in wafer_data.items():
    for step_id, data in steps.items():
        for t, row in enumerate(data):
            records.append({
                "wafer": wafer_id,
                "step": step_id,
                "time": t,
                **{f"sensor_{i+1}": row[i] for i in range(num_sensors)}
            })
df = pd.DataFrame(records)

# -------------------------------
# 2. Normalize sensor values
# -------------------------------
sensor_cols = [f"sensor_{i+1}" for i in range(num_sensors)]
scaler = StandardScaler()
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

# -------------------------------
# 3. Build wafer-series structure
# -------------------------------
wafer_series = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_data = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        avg_series = step_df[sensor_cols].mean(axis=1).values
        step_data.append(avg_series)
    wafer_series[wafer_id] = step_data

# -------------------------------
# 4. Simple DTW implementation
# -------------------------------
def simple_dtw(s1, s2):
    n, m = len(s1), len(s2)
    dtw_matrix = np.full((n+1, m+1), np.inf)
    dtw_matrix[0, 0] = 0
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(s1[i-1] - s2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1]
            )
    return dtw_matrix[n, m]

# -------------------------------
# 5. DTW-based anomaly index
# -------------------------------
reference_wafer = "W1"
reference_steps = wafer_series[reference_wafer]
wafer_distances_dtw = {}
for wafer_id, steps in wafer_series.items():
    if wafer_id == reference_wafer:
        wafer_distances_dtw[wafer_id] = 0.0
        continue
    dist_list = [simple_dtw(s1, s2) for s1, s2 in zip(reference_steps, steps)]
    wafer_distances_dtw[wafer_id] = np.mean(dist_list)

vals_dtw = np.array(list(wafer_distances_dtw.values()))
anomaly_dtw = (vals_dtw - vals_dtw.min()) / (vals_dtw.max() - vals_dtw.min())
anomaly_index_dtw = dict(zip(wafer_distances_dtw.keys(), anomaly_dtw))

# -------------------------------
# 6. PCA-based anomaly index
# -------------------------------
wafer_vectors = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_features = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        step_avg = step_df[sensor_cols].mean(axis=0).values
        step_features.append(step_avg)
    wafer_vector = np.concatenate(step_features)
    wafer_vectors[wafer_id] = wafer_vector

X = np.vstack(list(wafer_vectors.values()))
X_pca = PCA(n_components=2).fit_transform(X)
centroid = X_pca.mean(axis=0)
distances_pca = np.linalg.norm(X_pca - centroid, axis=1)
anomaly_pca = (distances_pca - distances_pca.min()) / (distances_pca.max() - distances_pca.min())
anomaly_index_pca = dict(zip(wafer_vectors.keys(), anomaly_pca))

# -------------------------------
# 7. Combine and display results
# -------------------------------
result_df = pd.DataFrame({
    "wafer": list(wafer_vectors.keys()),
    "anomaly_dtw": [anomaly_index_dtw[w] for w in wafer_vectors.keys()],
    "anomaly_pca": [anomaly_index_pca[w] for w in wafer_vectors.keys()]
})

result_df

,wafer,anomaly_dtw,anomaly_pca
0,W1,0.000000,0.901577
1,W2,0.609274,0.707444
2,W3,0.582469,0.320414
3,W4,0.671674,0.030757
4,W5,0.743909,0.000000
5,W6,0.780107,0.182083
6,W7,0.877893,0.629389
7,W8,1.000000,1.000000


In [43]:
# 재시도: matplotlib 제거하고 핵심 로직만 간단히 실행

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from random import randint

# 1. 데이터 생성
np.random.seed(42)
num_wafers = 8
num_steps = 21
num_sensors = 5

wafer_data = {}
for w in range(num_wafers):
    wafer_id = f"W{w+1}"
    wafer_data[wafer_id] = {}
    for s in range(num_steps):
        time_len = randint(30, 50)
        sensor_data = np.random.randn(time_len, num_sensors) + w * 0.1
        wafer_data[wafer_id][f"step_{s+1}"] = sensor_data

records = []
for wafer_id, steps in wafer_data.items():
    for step_id, data in steps.items():
        for t, row in enumerate(data):
            records.append({
                "wafer": wafer_id,
                "step": step_id,
                "time": t,
                **{f"sensor_{i+1}": row[i] for i in range(num_sensors)}
            })
df = pd.DataFrame(records)

# 2. 전처리
sensor_cols = [f"sensor_{i+1}" for i in range(num_sensors)]
scaler = StandardScaler()
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

# 3. wafer_series 구조 생성
wafer_series = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_data = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        avg_series = step_df[sensor_cols].mean(axis=1).values
        step_data.append(avg_series)
    wafer_series[wafer_id] = step_data

# 4. 간단한 DTW 함수
def simple_dtw(s1, s2):
    n, m = len(s1), len(s2)
    dtw_matrix = np.full((n+1, m+1), np.inf)
    dtw_matrix[0, 0] = 0
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(s1[i-1] - s2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1]
            )
    return dtw_matrix[n, m]

# 5. DTW 기반 anomaly index
reference_wafer = "W1"
reference_steps = wafer_series[reference_wafer]
wafer_distances_dtw = {}
for wafer_id, steps in wafer_series.items():
    if wafer_id == reference_wafer:
        wafer_distances_dtw[wafer_id] = 0.0
    else:
        dist_list = [simple_dtw(s1, s2) for s1, s2 in zip(reference_steps, steps)]
        wafer_distances_dtw[wafer_id] = np.mean(dist_list)
vals_dtw = np.array(list(wafer_distances_dtw.values()))
anomaly_dtw = (vals_dtw - vals_dtw.min()) / (vals_dtw.max() - vals_dtw.min())
anomaly_index_dtw = dict(zip(wafer_distances_dtw.keys(), anomaly_dtw))

# 6. PCA 기반 anomaly index
wafer_vectors = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_features = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        step_avg = step_df[sensor_cols].mean(axis=0).values
        step_features.append(step_avg)
    wafer_vector = np.concatenate(step_features)
    wafer_vectors[wafer_id] = wafer_vector
X = np.vstack(list(wafer_vectors.values()))
X_pca = PCA(n_components=2).fit_transform(X)
centroid = X_pca.mean(axis=0)
distances_pca = np.linalg.norm(X_pca - centroid, axis=1)
anomaly_pca = (distances_pca - distances_pca.min()) / (distances_pca.max() - distances_pca.min())
anomaly_index_pca = dict(zip(wafer_vectors.keys(), anomaly_pca))

# 7. 결과 통합
result_df = pd.DataFrame({
    "wafer": list(wafer_vectors.keys()),
    "anomaly_dtw": [anomaly_index_dtw[w] for w in wafer_vectors.keys()],
    "anomaly_pca": [anomaly_index_pca[w] for w in wafer_vectors.keys()]
})

In [46]:
# Re-run everything after kernel reset

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from random import randint
from scipy.stats import rankdata

# 1. Generate synthetic dataset
np.random.seed(42)
num_wafers = 8
num_steps = 21
num_sensors = 5

wafer_data = {}
for w in range(num_wafers):
    wafer_id = f"W{w+1}"
    wafer_data[wafer_id] = {}
    for s in range(num_steps):
        time_len = randint(30, 50)
        sensor_data = np.random.randn(time_len, num_sensors) + w * 0.1
        wafer_data[wafer_id][f"step_{s+1}"] = sensor_data

records = []
for wafer_id, steps in wafer_data.items():
    for step_id, data in steps.items():
        for t, row in enumerate(data):
            records.append({
                "wafer": wafer_id,
                "step": step_id,
                "time": t,
                **{f"sensor_{i+1}": row[i] for i in range(num_sensors)}
            })
df = pd.DataFrame(records)

# 2. Normalize sensor values
sensor_cols = [f"sensor_{i+1}" for i in range(num_sensors)]
scaler = StandardScaler()
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

# 3. Build wafer-series: step-wise time-series average per wafer
wafer_series = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_data = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        avg_series = step_df[sensor_cols].mean(axis=1).values
        step_data.append(avg_series)
    wafer_series[wafer_id] = step_data

# 4. Simple DTW function
def simple_dtw(s1, s2):
    n, m = len(s1), len(s2)
    dtw_matrix = np.full((n+1, m+1), np.inf)
    dtw_matrix[0, 0] = 0
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(s1[i-1] - s2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1]
            )
    return dtw_matrix[n, m]

# 5. DTW distance to reference wafer
reference_wafer = "W1"
reference_steps = wafer_series[reference_wafer]
wafer_distances_dtw = {}
for wafer_id, steps in wafer_series.items():
    if wafer_id == reference_wafer:
        wafer_distances_dtw[wafer_id] = 0.0
    else:
        dist_list = [simple_dtw(s1, s2) for s1, s2 in zip(reference_steps, steps)]
        wafer_distances_dtw[wafer_id] = np.mean(dist_list)

# 6. PCA-based distances
wafer_vectors = {}
for wafer_id in df['wafer'].unique():
    wafer_df = df[df['wafer'] == wafer_id]
    step_features = []
    for step_id in sorted(wafer_df['step'].unique(), key=lambda x: int(x.split("_")[1])):
        step_df = wafer_df[wafer_df['step'] == step_id]
        step_avg = step_df[sensor_cols].mean(axis=0).values
        step_features.append(step_avg)
    wafer_vector = np.concatenate(step_features)
    wafer_vectors[wafer_id] = wafer_vector

X = np.vstack(list(wafer_vectors.values()))
X_pca = PCA(n_components=2).fit_transform(X)
centroid = X_pca.mean(axis=0)
pca_distances = np.linalg.norm(X_pca - centroid, axis=1)

# 7. Percentile normalization function
def percentile_normalize(array):
    ranks = rankdata(array, method='average')  # 1 to N
    return (ranks - 1) / (len(ranks) - 1)       # → 0 to 1

# 8. Apply percentile normalization
wafer_ids = list(wafer_vectors.keys())
dtw_scores = np.array([wafer_distances_dtw[wid] for wid in wafer_ids])
pca_scores = pca_distances

dtw_index = percentile_normalize(dtw_scores)
pca_index = percentile_normalize(pca_scores)
hybrid_index = percentile_normalize(0.5 * dtw_scores + 0.5 * pca_scores)

# 9. Final result
result_df = pd.DataFrame({
    "wafer": wafer_ids,
    "dtw_percentile_index": dtw_index,
    "pca_percentile_index": pca_index,
    "hybrid_percentile_index": hybrid_index
})

result_df

,wafer,dtw_percentile_index,pca_percentile_index,hybrid_percentile_index
0,W1,0.000000,0.857143,0.000000
1,W2,0.142857,0.714286,0.428571
2,W3,0.285714,0.285714,0.142857
3,W4,0.428571,0.000000,0.285714
4,W5,0.571429,0.142857,0.571429
5,W6,0.714286,0.428571,0.714286
6,W7,0.857143,0.571429,0.857143
7,W8,1.000000,1.000000,1.000000
